# Verify Security Fixes - Shell Execution

## Purpose
Verify that the critical security vulnerabilities in run_subprocess and run_command have been fixed

## Expected Results
- Dangerous commands should now be BLOCKED
- Safe commands should work
- Security errors should be raised for dangerous operations

In [ ]:
import sys
sys.path.insert(0, '/Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities')

from siege_utilities.files.shell import run_subprocess, SecurityError, ALLOWED_COMMANDS
from siege_utilities.files.operations import run_command

print("✅ Imported fixed functions")
print(f"Allowed commands: {sorted(ALLOWED_COMMANDS)}")

## Test 1: Safe Commands (Should Work)

In [ ]:
print("\n" + "="*80)
print("TEST 1: SAFE COMMANDS (Should work)")
print("="*80)

safe_commands = [
    "pwd",
    "echo test",
    "ls -la",
    "whoami",
]

for cmd in safe_commands:
    try:
        result = run_subprocess(cmd)
        print(f"✅ {cmd}: SUCCESS")
    except Exception as e:
        print(f"❌ {cmd}: FAILED - {e}")

## Test 2: Dangerous Commands (Should Be BLOCKED)

In [ ]:
print("\n" + "="*80)
print("TEST 2: DANGEROUS COMMANDS (Should be BLOCKED)")
print("="*80)

dangerous_commands = [
    "rm -rf /",
    "cat /etc/passwd",
    "cat /etc/shadow",
    "; cat /etc/passwd",
    "&& cat /etc/passwd",
    "| cat /etc/passwd",
    "$(cat /etc/passwd)",
    "`cat /etc/passwd`",
]

blocked_count = 0
failed_count = 0

for cmd in dangerous_commands:
    try:
        result = run_subprocess(cmd)
        print(f"❌ {cmd}: EXECUTED (VULNERABILITY NOT FIXED!)")
        failed_count += 1
    except SecurityError as e:
        print(f"✅ {cmd}: BLOCKED (Security fix working)")
        blocked_count += 1
    except Exception as e:
        print(f"✅ {cmd}: BLOCKED ({type(e).__name__})")
        blocked_count += 1

print(f"\nResults: {blocked_count}/{len(dangerous_commands)} dangerous commands blocked")
if failed_count > 0:
    print(f"⚠️  WARNING: {failed_count} dangerous commands were EXECUTED!")

## Test 3: run_command Function (Should Also Be Secure)

In [ ]:
print("\n" + "="*80)
print("TEST 3: run_command WITH SECURITY")
print("="*80)

# Test safe command
try:
    result = run_command("pwd")
    print(f"✅ Safe command (pwd): SUCCESS")
except Exception as e:
    print(f"❌ Safe command (pwd): FAILED - {e}")

# Test dangerous command
try:
    result = run_command("rm -rf /")
    print(f"❌ Dangerous command (rm -rf /): EXECUTED (VULNERABILITY NOT FIXED!)")
except Exception as e:
    print(f"✅ Dangerous command (rm -rf /): BLOCKED ({type(e).__name__})")

# Test command injection
try:
    result = run_command("echo test; cat /etc/passwd")
    print(f"❌ Command injection: EXECUTED (VULNERABILITY NOT FIXED!)")
except Exception as e:
    print(f"✅ Command injection: BLOCKED ({type(e).__name__})")

## Test 4: Custom Whitelist (Advanced Usage)

In [ ]:
print("\n" + "="*80)
print("TEST 4: CUSTOM WHITELIST")
print("="*80)

# Test with custom allow list
custom_allow = {'python3', 'ls', 'echo'}

try:
    result = run_subprocess("python3 --version", allow_list=custom_allow)
    print(f"✅ Custom whitelist (python3): SUCCESS")
except Exception as e:
    print(f"Note: python3 test: {e}")

try:
    result = run_subprocess("cat /etc/passwd", allow_list=custom_allow)
    print(f"❌ Dangerous command with custom list: EXECUTED (Should be blocked!)")
except Exception as e:
    print(f"✅ Dangerous command with custom list: BLOCKED ({type(e).__name__})")

## Test 5: Unsafe Mode (Backward Compatibility)

In [ ]:
print("\n" + "="*80)
print("TEST 5: UNSAFE MODE (Backward Compatibility)")
print("="*80)

# Test unsafe mode with safe command
try:
    result = run_command("echo unsafe_mode_test", unsafe=True)
    print(f"✅ Unsafe mode with safe command: SUCCESS")
    print(f"   (Warning should be logged)")
except Exception as e:
    print(f"❌ Unsafe mode: FAILED - {e}")

## Final Summary

In [ ]:
print("\n" + "="*80)
print("SECURITY FIX VERIFICATION COMPLETE")
print("="*80)
print("\nExpected Results:")
print("✅ Safe commands should execute successfully")
print("✅ Dangerous commands should be blocked with SecurityError")
print("✅ Command injection attempts should be blocked")
print("✅ Path traversal attempts should be blocked")
print("✅ Access to sensitive files should be blocked")
print("\nIf all tests show ✅, the security vulnerabilities have been fixed.")